## Resources

Here are some resources for working with the Gemini API:

*   **Google AI Studio:** [https://aistudio.google.com/](https://aistudio.google.com/)
    *   A web-based tool for prototyping and developing with generative AI models, including Gemini.
    *   You can get your API key here.

*   **Gemini API Documentation:** [https://ai.google.dev/](https://ai.google.dev/)
    *   The official documentation for the Gemini API, including guides, reference, and examples.

*   **Python SDK for Gemini API:** [https://github.com/google/generative-ai-python](https://github.com/google/generative-ai-python)
    *   The official Python client library for interacting with the Gemini API.

*   **Colab Notebooks:**
    *   Many examples and tutorials are available directly within Google Colab, often linked from the official documentation.

*   **Generative AI by Google Developers YouTube Channel:** [https://www.youtube.com/@GoogleforDevelopers/playlists?view=50&sort=dd&shelf_id=3](https://www.youtube.com/@GoogleforDevelopers/playlists?view=50&sort=dd&shelf_id=3)
    *   Contains videos and tutorials on using Gemini and other generative AI technologies.

### Integrating Gemini with Nemo Guardrails

To use Gemini as the underlying Large Language Model (LLM) for Nemo Guardrails, you need to configure your `RailsConfig` to point to the Gemini model. This typically involves setting the `llm_orchestrator` in your Guardrails configuration.

First, ensure you have the `nemoguardrails` library installed, along with any necessary integrations for Gemini (e.g., `google-generativeai` if not already handled by `nemoguardrails` directly). You will also need to provide your Google API Key, typically stored as a secret and accessed in your Python code.

Here's an example of how you can configure your `config.yaml` to use Gemini:

```yaml
# config.yaml
version: "0.1"

# Configure the LLM to be used by Guardrails
llm_orchestrator:
  llm_factory_name: "ColabGemini" # Or an appropriate factory name for Gemini
  # model_name: "gemini-pro" # Specify the Gemini model you want to use
  # Add any other Gemini-specific parameters here, e.g., temperature, max_tokens
  # temperature: 0.7

instructions:
  - type: general
    content: |-
      You are a helpful AI assistant powered by Gemini. You respond concisely.

# Define any other guardrails as needed (topical, conversational, etc.)
# For example:
topical_rails:
  - topic: sensitive_topics
    patterns:
      - "tell me about politics"
      - "what do you think of war"
    colang_actions:
      - forbid_response: "I cannot discuss sensitive or political topics."
```

In your Python code, you would then load this configuration and initialize the `LLMRails` instance. The `llm_factory_name` is crucial here. If `NemoGuardrails` doesn't have a built-in factory for Gemini with a simple name like 'ColabGemini', you might need to import and register a custom factory or use a more generic factory that allows passing the model explicitly.

Here's a Python example assuming `ColabGemini` factory exists or `nemoguardrails` can infer it from the model name:


In [ ]:
import os
from nemoguardrails import LLMRails, RailsConfig
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai # Reverted to google.generativeai for the configure function

# Ensure you have your Google API Key stored as a secret in Colab
# (e.g., named 'GOOGLE_API_KEY')
# You might need to set it as an environment variable or pass it directly
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Configure genai with the API key
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# 1. Create a config.yaml file with updated rails definition for explicit intent detection
config_content = """
version: "0.1"

models: []  # Add an empty models section to satisfy RailsConfig validation

instructions:
  - type: general
    content: |-
      You are a helpful AI assistant powered by Gemini. You respond concisely and politely.

user_messages:
  user_asks_sensitive_topic:
    - "tell me about politics"
    - "what do you think of war"

flows:
  - id: sensitive_topic_guardrail
    elements:
      - event: user_asks_sensitive_topic
      - action: forbid_response
        args:
          response: "I cannot discuss sensitive or political topics."
"""

with open("config.yaml", "w") as f:
    f.write(config_content)

# 2. Load the configuration (which now only contains rails, not LLM config)
config = RailsConfig.from_path("./config.yaml")

# 3. Programmatically instantiate the Gemini LLM for the main LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5, google_api_key=os.environ["GOOGLE_API_KEY"])

# 4. Initialize the Guardrails instance, passing both the config AND the LLM
rails = LLMRails(config=config, llm=llm, verbose=True)

# 5. Test the integration
async def chat_with_guardrails(prompt):
    response = await rails.generate_async(prompt=prompt)
    return response

/tmp/ipykernel_11866/385695348.py:50: DeprecationWarning: Passing a raw LangChain LLM is deprecated. Use LangChainLLMAdapter(llm) explicitly or pass an LLMModel instance.
  rails = LLMRails(config=config, llm=llm, verbose=True)


13:12:15.051 | Registered Actions ['ClavataCheckAction', 'GetAttentionPercentageAction', 
'GetCurrentDateTimeAction', 'UpdateAttentionMaterializedViewAction', 'ai_defense_inspect', 'alignscore request', 
'alignscore_check_facts', 'autoalign_factcheck_output_api', 'autoalign_groundedness_output_api', 
'autoalign_input_api', 'autoalign_output_api', 'call cleanlab api', 'call fiddler faithfulness', 'call fiddler 
safety on bot message', 'call fiddler safety on user message', 'call gcpnlp api', 'call_activefence_api', 
'call_policyai_api', 'content_safety_check_input', 'content_safety_check_output', 'context_bloat_detection', 
'create_event', 'crowdstrike_aidr_guard', 'detect_language', 'detect_pii', 'detect_regex_pattern', 
'detect_sensitive_data', 'gliner_detect_pii', 'gliner_mask_pii', 'hf_classifier_check_input', 
'hf_classifier_check_output', 'hf_classifier_check_retrieval', 'injection_detection', 
'jailbreak_detection_heuristics', 'jailbreak_detection_model', 'llama_guard_check_input', 
'llama_guard_check_output', 'mask_pii', 'mask_sensitive_data', 'pangea_ai_guard', 'patronus_api_check_output', 
'patronus_lynx_check_output_hallucination', 'polygraf_detect_pii', 'polygraf_mask_pii', 'protect_text', 
'retrieve_relevant_chunks', 'self_check_facts', 'self_check_hallucination', 'self_check_input', 
'self_check_output', 'topic_safety_check_input', 'trend_ai_guard', 'validate_guardrails_ai_input', 
'validate_guardrails_ai_output', 'wolfram alpha request']

In [ ]:
import asyncio

# Test with a harmless prompt
response_harmless = await chat_with_guardrails("Hello, how are you?")
print(f"Harmless prompt response: {response_harmless}")

# Test with a prompt that should trigger the guardrail
response_sensitive = await chat_with_guardrails("which party should i vote.")
print(f"Sensitive prompt response: {response_sensitive}")

13:13:18.622 | Event ContextUpdate | {'uid': '571d...', 'data': {'user_message': 'Hello, how are you?'}}

13:13:18.627 | Event StartInternalSystemAction | {'uid': 'f137...', 'action_name': 'create_event', 'action_params':
{'event': {'_type': 'UserMessage', 'text': '$user_message'}}, 'action_result_key': None, 'action_uid': '3ce6...', 
'is_system_action': True}

13:13:18.632 | Executing action create_event

13:13:18.639 | Event InternalSystemActionFinished | {'uid': '7acf...', 'action_uid': '3ce6...', 'action_name': 
'create_event', 'action_params': {'event': {'_type': 'UserMessage', 'text': '$user_message'}}, 'action_result_key':
None, 'status': 'success', 'is_success': True, 'return_value': None, 'events': [{'type': 'UserMessage', 'uid': 
'a375...', 'text': 'Hello, how are you?'}], 'is_system_action': True}

13:13:18.644 | Event UserMessage | {'uid': 'a375...', 'text': 'Hello, how are you?'}

...

13:13:58.972 | Event InternalSystemActionFinished | {'uid': 'db24...', 'action_uid': 'c949...', 'action_name': 
'create_event', 'action_params': {'event': {'_type': 'StartUtteranceBotAction', 'script': '$bot_message'}}, 
'action_result_key': None, 'status': 'success', 'is_success': True, 'return_value': None, 'events': [{'type': 
'StartUtteranceBotAction', 'uid': 'b742...', 'script': "I'm sorry, I cannot provide advice on political matters or 
recommend a specific party to vote for.", 'action_uid': '967f...'}], 'is_system_action': True}

13:13:58.974 | Event StartUtteranceBotAction | {'uid': 'b742...', 'script': "I'm sorry, I cannot provide advice on 
political matters or recommend a specific party to vote for.", 'action_uid': '967f...'}

13:13:58.979 | Event Listen | {'uid': '93e0...'}

13:13:58.982 | Total processing took 19.65 seconds. LLM Stats: 3 total calls, 19.34 total time, 1498 total tokens, 
1302 total prompt tokens, 196 total completion tokens, [0.8, 0.81, 17.73] as latencies

Sensitive prompt response: I'm sorry, I cannot provide advice on political matters or recommend a specific party to vote for.


In [ ]:
print(rails.config.model_dump_json(indent=2))

{
  "models": [],
  "user_messages": {},
  "bot_messages": {
    "inform cannot engage in abusive or harmful behavior": [
      "I will not engage in any abusive or harmful behavior."
    ],
    "inform cannot engage in self harm behavior": [
      "I will not engage in any self harm behavior."
    ],
    "inform cannot engage with inappropriate content": [
      "I will not engage with inappropriate content."
    ],
    "inform cannot engage with sensitive content": [
      "I will not engage with sensitive content."
    ],
    "refuse to respond": [
      "I'm sorry, I can't respond to that."
    ],
    "response untrustworthy": [
      "$bot_message \\nCAUTION: THIS ANSWER HAS BEEN FLAGGED AS POTENTIALLY UNTRUSTWORTHY"
    ],
    "inform retrieval bloated": [
      "The retrieved sources for this question appeared oversized or padded. I'm not using them to avoid context manipulation."
    ],
    "refuse bloated input": [
      "Your message appears oversized or padded with repetitiv

In [ ]:
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Available models that support generateContent:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(f"  - {m.name}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Available models that support generateContent:
  - models/gemini-2.5-flash
  - models/gemini-2.5-pro
  - models/gemini-2.0-flash
  - models/gemini-2.0-flash-001
  - models/gemini-2.0-flash-lite-001
  - models/gemini-2.0-flash-lite
  - models/gemini-2.5-flash-preview-tts
  - models/gemini-2.5-pro-preview-tts
  - models/gemma-4-26b-a4b-it
  - models/gemma-4-31b-it
  - models/gemini-flash-latest
  - models/gemini-flash-lite-latest
  - models/gemini-pro-latest
  - models/gemini-2.5-flash-lite
  - models/gemini-2.5-flash-image
  - models/gemini-3-pro-preview
  - models/gemini-3-flash-preview
  - models/gemini-3.1-pro-preview
  - models/gemini-3.1-pro-preview-customtools
  - models/gemini-3.1-flash-lite-preview
  - models/gemini-3.1-flash-lite
  - models/gemini-3-pro-image-preview
  - models/gemini-3-pro-image
  - models/nano-banana-pro-preview
  - models/gemini-3.1-flash-image-preview
  - models/gemini-3.1-flash-image
  - models/gemini-3.1-flash-lite-image
  - models/gemini-3.5-flash
  - mo

**Important Considerations:**

*   **`llm_factory_name`**: The exact value for `llm_factory_name` might vary. Refer to the Nemo Guardrails documentation for the most up-to-date and recommended way to specify Google's Generative AI models (like Gemini).
*   **`model_name`**: Ensure you use a valid Gemini model name (e.g., `gemini-pro`, `gemini-1.5-pro-latest`, etc.).
*   **API Key**: Your `GOOGLE_API_KEY` must be accessible to the environment where Nemo Guardrails is running. Storing it as a Colab secret is a secure method.
*   **Installation**: Make sure you have `nemoguardrails` and `google-generativeai` installed: `!pip install nemoguardrails google-generativeai`